# Full Multitask XGBoost Pipeline

This notebook combines the strongest current modeling iterations into one end-to-end workflow:

1. Stage 1: track-level classification for `will_spread`
2. Stage 2: row-level country ranking for `did_enter_within_60d`
3. Stage 3: row-level regression for `days_to_entry`

The end-to-end pipeline uses stage 1 to decide which tracks should be forwarded, stage 2 to rank likely next countries, and stage 3 to estimate how many days it will take those countries to chart.


In [ ]:
from pathlib import Path
import json
import os
import tempfile
import warnings

os.environ.setdefault('MPLCONFIGDIR', tempfile.mkdtemp(prefix='matplotlib-'))

import duckdb
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    fbeta_score,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

try:
    import xgboost as xgb
except ImportError as exc:
    raise ImportError('Install xgboost first, e.g. `pip install -r requirements.txt`.') from exc

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'datasets').exists() and (candidate / 'requirements.txt').exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root from {start}. Expected a parent containing 'datasets/' and 'requirements.txt'."
    )


ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = ROOT / 'datasets' / 'processed' / 'v3_features'
MODEL_DIR = ROOT / 'artifacts' / 'models' / 'xgboost_multitask_pipeline'
EVAL_DIR = ROOT / 'artifacts' / 'evaluations' / 'xgboost_multitask_pipeline'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train.parquet'
VAL_PATH = DATA_DIR / 'val.parquet'
TEST_PATH = DATA_DIR / 'test.parquet'
RANKER_SUMMARY_PATH = ROOT / 'artifacts' / 'models' / 'xgboost_ranker_temporal_tuned' / 'training_summary.json'

assert TRAIN_PATH.exists(), f'Missing training split: {TRAIN_PATH}'
assert VAL_PATH.exists(), f'Missing validation split: {VAL_PATH}'
assert TEST_PATH.exists(), f'Missing test split: {TEST_PATH}'

RANDOM_STATE = 42
RUN_FULL_SPLITS = True
DEBUG_MAX_TRAIN_TRACKS = 2000
DEBUG_MAX_VAL_TRACKS = 1000
DEBUG_MAX_TEST_TRACKS = 1000
TOP_K = 5
PRIMARY_PRECISION_FLOOR = 0.20
SECONDARY_PRECISION_FLOORS = [0.25]
RUN_RANKER_RETUNING = False
RUN_REGRESSOR_TUNING = False
USE_PIPELINE_AWARE_THRESHOLDS = True
RUN_SHAP = False
SHAP_SAMPLE_TRACKS = 1000

TRAIN_MAX_TRACKS = None if RUN_FULL_SPLITS else DEBUG_MAX_TRAIN_TRACKS
VAL_MAX_TRACKS = None if RUN_FULL_SPLITS else DEBUG_MAX_VAL_TRACKS
TEST_MAX_TRACKS = None if RUN_FULL_SPLITS else DEBUG_MAX_TEST_TRACKS
RUN_MODE = 'full' if RUN_FULL_SPLITS else 'debug_sampled'
BUSINESS_PRECISION_FLOORS = [PRIMARY_PRECISION_FLOOR, *SECONDARY_PRECISION_FLOORS]

STAGE1_FIXED_PARAMS = {
    'scale_pos_weight': 1.0,
    'max_depth': 6,
    'min_child_weight': 5,
}
STAGE1_N_ESTIMATORS = 500
STAGE1_EARLY_STOPPING_ROUNDS = 30
STAGE2_FALLBACK_PARAMS = {
    'learning_rate': 0.07303585909985359,
    'max_depth': 9,
    'min_child_weight': 3.0893050901115746,
    'subsample': 0.8729982015899902,
    'colsample_bytree': 0.5698762418046549,
    'gamma': 0.9995410123755416,
    'reg_alpha': 0.0001088457180562647,
    'reg_lambda': 6.467502376208379,
}
STAGE2_FALLBACK_FINAL_N_ESTIMATORS = 240
STAGE2_EARLY_STOPPING_ROUNDS = 50
STAGE2_TUNING_N_ESTIMATORS = 2000
STAGE2_FINAL_N_ESTIMATOR_BUFFER = 1.10
STAGE2_TUNING_TRIALS = 8
STAGE2_TIME_BLOCKS = 5

STAGE3_DEFAULT_PARAMS = {
    'learning_rate': 0.05,
    'max_depth': 6,
    'min_child_weight': 10,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_lambda': 1.0,
    'reg_alpha': 0.0,
}
STAGE3_PARAM_GRID = [
    {'learning_rate': 0.05, 'max_depth': 4, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0},
    {'learning_rate': 0.05, 'max_depth': 6, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0},
    {'learning_rate': 0.05, 'max_depth': 6, 'min_child_weight': 10, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0},
    {'learning_rate': 0.03, 'max_depth': 6, 'min_child_weight': 10, 'subsample': 0.85, 'colsample_bytree': 0.85, 'reg_lambda': 1.0, 'reg_alpha': 0.0},
    {'learning_rate': 0.05, 'max_depth': 8, 'min_child_weight': 10, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0},
]
STAGE3_N_ESTIMATORS = 800
STAGE3_EARLY_STOPPING_ROUNDS = 30
STAGE3_TARGET_TRANSFORMS = ['identity', 'log1p']

print({
    'run_mode': RUN_MODE,
    'top_k': TOP_K,
    'primary_precision_floor': PRIMARY_PRECISION_FLOOR,
    'secondary_precision_floors': SECONDARY_PRECISION_FLOORS,
    'run_ranker_retuning': RUN_RANKER_RETUNING,
    'run_regressor_tuning': RUN_REGRESSOR_TUNING,
    'use_pipeline_aware_thresholds': USE_PIPELINE_AWARE_THRESHOLDS,
    'run_shap': RUN_SHAP,
    'train_max_tracks': TRAIN_MAX_TRACKS,
    'val_max_tracks': VAL_MAX_TRACKS,
    'test_max_tracks': TEST_MAX_TRACKS,
})


In [ ]:
con = duckdb.connect()

FEATURE_EXCLUDE = ['track_id', 'observation_time', 'target_country', 'did_enter_within_60d', 'days_to_entry']
TARGET_SPECIFIC_COLS = [
    'artist_prior_success_in_target',
    'target_population',
    'target_avg_daily_streams',
    'target_new_entry_rate_30d',
    'target_continent_africa',
    'target_continent_asia',
    'target_continent_europe',
    'target_continent_north_america',
    'target_continent_oceania',
    'target_continent_south_america',
    'cultural_dist_min',
    'cultural_dist_missing',
    'same_language_flag',
    'same_continent_flag',
    'neighbor_entered_count',
]
EXCLUDE_FROM_TRACK_LEVEL = {'target_country', 'did_enter_within_60d', 'days_to_entry'}


def load_row_level_split(path: Path, max_tracks: int | None = None) -> pd.DataFrame:
    parquet_path = path.as_posix()
    if max_tracks is None:
        query = f"SELECT * FROM read_parquet('{parquet_path}')"
    else:
        query = f"""
            WITH sampled_tracks AS (
                SELECT track_id
                FROM read_parquet('{parquet_path}')
                GROUP BY track_id
                ORDER BY hash(track_id)
                LIMIT {int(max_tracks)}
            )
            SELECT d.*
            FROM read_parquet('{parquet_path}') d
            JOIN sampled_tracks st USING (track_id)
        """
    df = con.execute(query).fetchdf()
    df['observation_time'] = pd.to_datetime(df['observation_time'])
    return df


def load_track_level_split(path: Path, max_tracks: int | None = None) -> pd.DataFrame:
    parquet_path = path.as_posix()
    schema_df = con.execute(f"SELECT * FROM read_parquet('{parquet_path}') LIMIT 0").fetchdf()
    raw_cols = list(schema_df.columns)
    constant_cols = [c for c in raw_cols if c not in EXCLUDE_FROM_TRACK_LEVEL and c not in TARGET_SPECIFIC_COLS]
    rank_cols = [c for c in raw_cols if c.startswith('rank_')]

    source_expr = f"read_parquet('{parquet_path}')"
    if max_tracks is None:
        source_query = source_expr
    else:
        source_query = f"""
            (
                WITH sampled_tracks AS (
                    SELECT track_id
                    FROM {source_expr}
                    GROUP BY track_id
                    ORDER BY hash(track_id)
                    LIMIT {int(max_tracks)}
                )
                SELECT d.*
                FROM {source_expr} d
                JOIN sampled_tracks st USING (track_id)
            )
        """

    select_parts = [
        'track_id',
        'MAX(CAST(did_enter_within_60d AS INTEGER)) AS will_spread',
        'MIN(CASE WHEN did_enter_within_60d = 1 THEN days_to_entry END) AS min_days_to_spread',
        'COUNT(*) AS candidate_count',
    ]
    select_parts.extend([f'ANY_VALUE({col}) AS {col}' for col in constant_cols if col != 'track_id'])
    for col in TARGET_SPECIFIC_COLS:
        select_parts.append(f'AVG({col}) AS {col}_mean')
        select_parts.append(f'MAX({col}) AS {col}_max')

    query = f"""
        SELECT
            {', '.join(select_parts)}
        FROM {source_query}
        GROUP BY track_id
        ORDER BY track_id
    """
    df = con.execute(query).fetchdf()
    df['observation_time'] = pd.to_datetime(df['observation_time'])
    df['origin_country_count_at_obs'] = (df[rank_cols].fillna(0) > 0).sum(axis=1)
    return df


def make_feature_matrix(df: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series) -> pd.DataFrame:
    return df[feature_cols].copy().fillna(fill_values)


def safe_roc_auc(y_true: pd.Series, y_score: np.ndarray) -> float | None:
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return None
    return float(roc_auc_score(y_true, y_score))


def safe_average_precision(y_true: pd.Series, y_score: np.ndarray) -> float | None:
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return None
    return float(average_precision_score(y_true, y_score))


def binary_metrics(y_true: pd.Series, y_score: np.ndarray) -> dict:
    clipped = np.clip(y_score, 1e-6, 1 - 1e-6)
    return {
        'roc_auc': safe_roc_auc(y_true, y_score),
        'average_precision': safe_average_precision(y_true, y_score),
        'log_loss': float(log_loss(y_true, clipped, labels=[0, 1])),
    }


def build_threshold_table(y_true: pd.Series, y_prob: np.ndarray, beta: float = 2.0) -> pd.DataFrame:
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    rows = []
    beta_sq = beta ** 2
    for p, r, t in zip(precision[:-1], recall[:-1], thresholds):
        denom = p + r
        f1 = 0.0 if denom == 0 else 2 * p * r / denom
        fbeta = 0.0 if (beta_sq * p + r) == 0 else (1 + beta_sq) * p * r / (beta_sq * p + r)
        predicted_positive_rate = float((y_prob >= t).mean())
        rows.append({
            'threshold': float(t),
            'precision': float(p),
            'recall': float(r),
            'f1': float(f1),
            f'f{beta}': float(fbeta),
            'predicted_positive_rate': predicted_positive_rate,
            'flagged_per_1000_tracks': predicted_positive_rate * 1000.0,
        })
    return pd.DataFrame(rows).sort_values(['threshold']).reset_index(drop=True)


def choose_recall_threshold(y_true: pd.Series, y_prob: np.ndarray, precision_floor: float, beta: float = 2.0) -> tuple[float, pd.DataFrame, str]:
    threshold_df = build_threshold_table(y_true, y_prob, beta=beta)
    eligible = threshold_df[threshold_df['precision'] >= precision_floor].copy()
    if not eligible.empty:
        selected = eligible.sort_values(['recall', 'precision', 'threshold'], ascending=[False, False, True]).iloc[0]
        reason = f'highest recall with precision >= {precision_floor:.2f}'
    else:
        score_col = f'f{beta}'
        selected = threshold_df.sort_values([score_col, 'recall', 'precision'], ascending=[False, False, False]).iloc[0]
        reason = f'fallback to best {score_col} because no threshold met precision floor {precision_floor:.2f}'
    return float(selected['threshold']), threshold_df, reason


def binary_summary(y_true: pd.Series, y_prob: np.ndarray, threshold: float, beta: float = 2.0) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    clipped = np.clip(y_prob, 1e-6, 1 - 1e-6)
    return {
        'roc_auc': safe_roc_auc(y_true, y_prob),
        'average_precision': safe_average_precision(y_true, y_prob),
        'log_loss': float(log_loss(y_true, clipped, labels=[0, 1])),
        'brier_score': float(brier_score_loss(y_true, clipped)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        f'f{beta}': float(fbeta_score(y_true, y_pred, beta=beta, zero_division=0)),
        'positive_rate': float(np.mean(y_true)),
        'predicted_positive_rate': float(np.mean(y_pred)),
        'flagged_per_1000_tracks': float(np.mean(y_pred) * 1000.0),
    }


def build_business_threshold_summary(y_true: pd.Series, y_prob: np.ndarray, floors: list[float], beta: float = 2.0) -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    rows = []
    threshold_tables = []
    threshold_map = {}
    for floor in floors:
        threshold, threshold_df, reason = choose_recall_threshold(y_true, y_prob, precision_floor=floor, beta=beta)
        metrics = binary_summary(y_true, y_prob, threshold=threshold, beta=beta)
        rows.append({
            'precision_floor': float(floor),
            'threshold': float(threshold),
            'selection_reason': reason,
            **metrics,
        })
        threshold_map[float(floor)] = float(threshold)
        threshold_df = threshold_df.copy()
        threshold_df['precision_floor'] = float(floor)
        threshold_tables.append(threshold_df)
    
    return pd.DataFrame(rows), threshold_map, pd.concat(threshold_tables, ignore_index=True)


def summarize_gated_track_metrics(stage1_scored_df: pd.DataFrame, ranker_track_metrics_df: pd.DataFrame, threshold: float, k: int = 5) -> dict:
    merged = stage1_scored_df[['track_id', 'score']].merge(ranker_track_metrics_df, on='track_id', how='inner')
    positive_mask = merged['positives'] > 0
    flagged = merged['score'] >= float(threshold)
    total_tracks = len(merged)
    positive_tracks = int(positive_mask.sum())
    recall_col = f'recall@{k}'
    hit_col = f'hit_rate@{k}'
    ndcg_col = f'ndcg@{k}'
    map_col = f'map@{k}'
    return {
        'flagged_tracks': int(flagged.sum()),
        f'pipeline_precision@{k}': float(merged.loc[flagged, 'top_k_hits'].sum() / max(total_tracks * k, 1)),
        f'pipeline_recall@{k}': float(merged.loc[positive_mask & flagged, recall_col].fillna(0.0).sum() / max(positive_tracks, 1)),
        f'pipeline_hit_rate@{k}': float(merged.loc[positive_mask & flagged, hit_col].fillna(0.0).sum() / max(positive_tracks, 1)),
        f'pipeline_ndcg@{k}': float(merged.loc[positive_mask & flagged, ndcg_col].fillna(0.0).sum() / max(positive_tracks, 1)),
        f'pipeline_map@{k}': float(merged.loc[positive_mask & flagged, map_col].fillna(0.0).sum() / max(positive_tracks, 1)),
    }


def build_pipeline_threshold_candidates(stage1_scored_df: pd.DataFrame, ranker_track_metrics_df: pd.DataFrame, k: int = 5) -> pd.DataFrame:
    merged = stage1_scored_df[['track_id', 'score', 'will_spread']].merge(
        ranker_track_metrics_df[['track_id', 'positives', 'top_k_hits', f'recall@{k}', f'hit_rate@{k}', f'ndcg@{k}', f'map@{k}']],
        on='track_id',
        how='inner',
    )
    merged = merged.sort_values(['score', 'track_id'], ascending=[False, True]).reset_index(drop=True)
    positive_tracks = int((merged['positives'] > 0).sum())
    total_tracks = len(merged)
    merged['flagged_tracks'] = np.arange(1, total_tracks + 1)
    merged['spreaders_flagged'] = merged['will_spread'].astype(int).cumsum()
    merged['top_k_hits_flagged'] = merged['top_k_hits'].fillna(0.0).cumsum()
    for metric_col in [f'recall@{k}', f'hit_rate@{k}', f'ndcg@{k}', f'map@{k}']:
        value_col = metric_col.replace('@', '_value_')
        merged[value_col] = np.where(merged['positives'] > 0, merged[metric_col].fillna(0.0), 0.0)
        merged[f'cum_{value_col}'] = merged[value_col].cumsum()

    grouped = merged.groupby('score', as_index=False).agg(
        threshold=('score', 'first'),
        flagged_tracks=('flagged_tracks', 'max'),
        spreaders_flagged=('spreaders_flagged', 'max'),
        top_k_hits_flagged=('top_k_hits_flagged', 'max'),
        recall_value_sum=(f'cum_recall_value_{k}', 'max'),
        hit_rate_value_sum=(f'cum_hit_rate_value_{k}', 'max'),
        ndcg_value_sum=(f'cum_ndcg_value_{k}', 'max'),
        map_value_sum=(f'cum_map_value_{k}', 'max'),
    )
    grouped['spread_precision'] = grouped['spreaders_flagged'] / grouped['flagged_tracks']
    grouped['spread_recall'] = grouped['spreaders_flagged'] / max(positive_tracks, 1)
    grouped[f'pipeline_precision@{k}'] = grouped['top_k_hits_flagged'] / max(total_tracks * k, 1)
    grouped[f'pipeline_recall@{k}'] = grouped['recall_value_sum'] / max(positive_tracks, 1)
    grouped[f'pipeline_hit_rate@{k}'] = grouped['hit_rate_value_sum'] / max(positive_tracks, 1)
    grouped[f'pipeline_ndcg@{k}'] = grouped['ndcg_value_sum'] / max(positive_tracks, 1)
    grouped[f'pipeline_map@{k}'] = grouped['map_value_sum'] / max(positive_tracks, 1)
    return grouped.sort_values('threshold').reset_index(drop=True)


def choose_pipeline_thresholds(stage1_scored_df: pd.DataFrame, ranker_track_metrics_df: pd.DataFrame, floors: list[float], k: int = 5) -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    candidate_df = build_pipeline_threshold_candidates(stage1_scored_df, ranker_track_metrics_df, k=k)
    rows = []
    threshold_map = {}
    for floor in floors:
        eligible = candidate_df[candidate_df['spread_precision'] >= float(floor)].copy()
        if not eligible.empty:
            selected = eligible.sort_values([f'pipeline_recall@{k}', f'pipeline_hit_rate@{k}', f'pipeline_ndcg@{k}', 'threshold'], ascending=[False, False, False, True]).iloc[0]
            reason = f'highest pipeline recall@{k} with spread precision >= {float(floor):.2f}'
        else:
            selected = candidate_df.sort_values([f'pipeline_recall@{k}', f'pipeline_hit_rate@{k}', f'pipeline_ndcg@{k}', 'threshold'], ascending=[False, False, False, True]).iloc[0]
            reason = f'fallback to best pipeline recall@{k} because no threshold met spread precision floor {float(floor):.2f}'
        rows.append({
            'precision_floor': float(floor),
            'threshold': float(selected['threshold']),
            'selection_reason': reason,
            'spread_precision': float(selected['spread_precision']),
            'spread_recall': float(selected['spread_recall']),
            f'pipeline_precision@{k}': float(selected[f'pipeline_precision@{k}']),
            f'pipeline_recall@{k}': float(selected[f'pipeline_recall@{k}']),
            f'pipeline_hit_rate@{k}': float(selected[f'pipeline_hit_rate@{k}']),
            f'pipeline_ndcg@{k}': float(selected[f'pipeline_ndcg@{k}']),
            f'pipeline_map@{k}': float(selected[f'pipeline_map@{k}']),
            'flagged_tracks': int(selected['flagged_tracks']),
        })
        threshold_map[float(floor)] = float(selected['threshold'])
    return pd.DataFrame(rows), threshold_map, candidate_df


def compute_candidate_stats(scored_df: pd.DataFrame) -> dict:
    per_track = scored_df.groupby('track_id').agg(
        candidates=('target_country', 'size'),
        positives=('did_enter_within_60d', 'sum'),
    )
    positive_mask = per_track['positives'] > 0
    return {
        'tracks': int(per_track.shape[0]),
        'positive_tracks': int(positive_mask.sum()),
        'avg_candidates_per_track': float(per_track['candidates'].mean()),
        'avg_future_countries_per_track': float(per_track['positives'].mean()),
        'avg_future_countries_per_positive_track': float(per_track.loc[positive_mask, 'positives'].mean()) if positive_mask.any() else None,
    }


def ranking_metrics(scored_df: pd.DataFrame, k: int = 5) -> tuple[dict, pd.DataFrame]:
    rows = []
    for track_id, group in scored_df.groupby('track_id', sort=False):
        group = group.sort_values(['score', 'tie_break'], ascending=[False, False]).reset_index(drop=True)
        labels = group['did_enter_within_60d'].to_numpy(dtype=int)
        positives = int(labels.sum())
        top = group.head(k)
        top_labels = top['did_enter_within_60d'].to_numpy(dtype=int)
        hits = int(top_labels.sum())

        precision = hits / k
        recall = hits / positives if positives else np.nan
        hit_rate = float(hits > 0) if positives else np.nan

        discounts = np.log2(np.arange(2, len(top_labels) + 2))
        dcg = float(((2 ** top_labels - 1) / discounts).sum())
        ideal = np.sort(labels)[::-1][: len(top_labels)]
        idcg = float(((2 ** ideal - 1) / np.log2(np.arange(2, len(ideal) + 2))).sum())
        ndcg = dcg / idcg if idcg > 0 else np.nan

        ap_accum = 0.0
        running_hits = 0
        for rank, rel in enumerate(top_labels, start=1):
            if rel:
                running_hits += 1
                ap_accum += running_hits / rank
        map_k = ap_accum / min(positives, k) if positives else np.nan

        rows.append({
            'track_id': track_id,
            'positives': positives,
            'top_k_hits': hits,
            f'precision@{k}': precision,
            f'recall@{k}': recall,
            f'hit_rate@{k}': hit_rate,
            f'ndcg@{k}': ndcg,
            f'map@{k}': map_k,
        })

    metric_df = pd.DataFrame(rows)
    positive_mask = metric_df['positives'] > 0
    summary = {
        'tracks': int(metric_df.shape[0]),
        'positive_tracks': int(positive_mask.sum()),
        f'precision@{k}': float(metric_df[f'precision@{k}'].mean()),
        f'recall@{k}': float(metric_df.loc[positive_mask, f'recall@{k}'].mean()) if positive_mask.any() else None,
        f'hit_rate@{k}': float(metric_df.loc[positive_mask, f'hit_rate@{k}'].mean()) if positive_mask.any() else None,
        f'ndcg@{k}': float(metric_df.loc[positive_mask, f'ndcg@{k}'].mean()) if positive_mask.any() else None,
        f'map@{k}': float(metric_df.loc[positive_mask, f'map@{k}'].mean()) if positive_mask.any() else None,
        'mean_future_countries_per_track': float(metric_df['positives'].mean()),
    }
    return summary, metric_df


def evaluate_ranked_candidates(scored_df: pd.DataFrame, k: int = 5) -> tuple[dict, pd.DataFrame]:
    candidate_stats = compute_candidate_stats(scored_df)
    binary = binary_metrics(scored_df['did_enter_within_60d'], scored_df['score'].to_numpy())
    ranking_all, track_metrics = ranking_metrics(scored_df, k=k)
    positive_track_metrics = track_metrics[track_metrics['positives'] > 0].copy()
    positive_summary = {
        'tracks': int(positive_track_metrics.shape[0]),
        'avg_future_countries_per_positive_track': float(positive_track_metrics['positives'].mean()) if not positive_track_metrics.empty else None,
        f'avg_top_{k}_hits_per_positive_track': float(positive_track_metrics['top_k_hits'].mean()) if not positive_track_metrics.empty else None,
        f'recall@{k}': float(positive_track_metrics[f'recall@{k}'].mean()) if not positive_track_metrics.empty else None,
        f'hit_rate@{k}': float(positive_track_metrics[f'hit_rate@{k}'].mean()) if not positive_track_metrics.empty else None,
        f'ndcg@{k}': float(positive_track_metrics[f'ndcg@{k}'].mean()) if not positive_track_metrics.empty else None,
        f'map@{k}': float(positive_track_metrics[f'map@{k}'].mean()) if not positive_track_metrics.empty else None,
    }
    return {
        'binary': binary,
        'candidate_stats': candidate_stats,
        'ranking_all_tracks': ranking_all,
        'ranking_positive_tracks': positive_summary,
    }, track_metrics


def score_naive_popularity(df: pd.DataFrame) -> pd.DataFrame:
    scored = df[['track_id', 'observation_time', 'target_country', 'did_enter_within_60d', 'days_to_entry', 'target_avg_daily_streams', 'target_new_entry_rate_30d']].copy()
    primary = scored['target_avg_daily_streams'].fillna(0.0)
    tie_break = scored['target_new_entry_rate_30d'].fillna(0.0)
    raw_score = primary + tie_break * 1e-6
    score_min = float(raw_score.min())
    score_max = float(raw_score.max())
    if score_max > score_min:
        scored['score'] = (raw_score - score_min) / (score_max - score_min)
    else:
        scored['score'] = 0.5
    scored['tie_break'] = tie_break
    scored['model_name'] = 'naive_popularity_baseline'
    return scored


def feature_category(name: str) -> str:
    if name.startswith('rank_') or name in {'track_in_viral50_at_obs', 'candidate_count', 'origin_country_count_at_obs'}:
        return 'current_footprint'
    if name.startswith('artist_') or name == 'multi_artist_flag':
        return 'artist_history'
    if name.startswith('target_'):
        return 'target_country_priors'
    if name in {'cultural_dist_min', 'cultural_dist_missing', 'same_language_flag', 'same_continent_flag', 'neighbor_entered_count'}:
        return 'origin_target_relationship'
    if name.endswith('_mean') or name.endswith('_max'):
        return 'aggregates'
    if name.startswith('af_') or name in {'duration_ms', 'explicit', 'days_since_release', 'is_friday_release'}:
        return 'audio_track_metadata'
    if name.startswith('observation_'):
        return 'temporal'
    return 'other'


def load_frozen_ranker_config(summary_path: Path) -> tuple[dict, int]:
    if summary_path.exists():
        data = json.loads(summary_path.read_text())
        params = data.get('tuning', {}).get('best_params')
        final_n_estimators = data.get('config', {}).get('final_n_estimators')
        if params:
            return params, int(final_n_estimators or STAGE2_FALLBACK_FINAL_N_ESTIMATORS)
    return STAGE2_FALLBACK_PARAMS.copy(), STAGE2_FALLBACK_FINAL_N_ESTIMATORS


def prepare_ranker_inputs(df: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series):
    ordered = df.sort_values(['track_id', 'target_country']).reset_index(drop=True)
    X = make_feature_matrix(ordered, feature_cols, fill_values)
    y = ordered['did_enter_within_60d'].astype(float).to_numpy()
    group = ordered.groupby('track_id', sort=False).size().to_numpy()
    return ordered, X, y, group


def make_ranker(params: dict, n_estimators: int = STAGE2_TUNING_N_ESTIMATORS, early_stopping_rounds: int | None = STAGE2_EARLY_STOPPING_ROUNDS):
    init_kwargs = {
        'objective': 'rank:ndcg',
        'eval_metric': 'ndcg@5',
        'tree_method': 'hist',
        'n_estimators': n_estimators,
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        **params,
    }
    if early_stopping_rounds is not None:
        init_kwargs['early_stopping_rounds'] = early_stopping_rounds
    return xgb.XGBRanker(**init_kwargs)


def score_ranker(model, df: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series, model_name: str) -> pd.DataFrame:
    ordered = df.sort_values(['track_id', 'target_country']).reset_index(drop=True)
    X = make_feature_matrix(ordered, feature_cols, fill_values)
    scores = model.predict(X)
    scored = ordered[['track_id', 'observation_time', 'target_country', 'did_enter_within_60d', 'days_to_entry', 'target_avg_daily_streams', 'target_new_entry_rate_30d']].copy()
    scored['score'] = scores
    scored['tie_break'] = scored['target_new_entry_rate_30d'].fillna(0.0)
    scored['model_name'] = model_name
    return scored


def make_temporal_folds(train_df: pd.DataFrame, time_blocks: int = 5) -> list[dict]:
    track_times = train_df[['track_id', 'observation_time']].drop_duplicates().sort_values(['observation_time', 'track_id']).reset_index(drop=True)
    track_times['time_block'] = pd.qcut(track_times.index, q=time_blocks, labels=False, duplicates='drop')
    folds = []
    unique_blocks = sorted(track_times['time_block'].dropna().astype(int).unique())
    for fold_idx, block in enumerate(unique_blocks[2:], start=1):
        train_track_ids = track_times.loc[track_times['time_block'] < block, 'track_id'].tolist()
        val_track_ids = track_times.loc[track_times['time_block'] == block, 'track_id'].tolist()
        if not train_track_ids or not val_track_ids:
            continue
        train_times = track_times.loc[track_times['time_block'] < block, 'observation_time']
        val_times = track_times.loc[track_times['time_block'] == block, 'observation_time']
        folds.append({
            'fold': fold_idx,
            'train_track_ids': train_track_ids,
            'val_track_ids': val_track_ids,
            'train_tracks': len(train_track_ids),
            'val_tracks': len(val_track_ids),
            'train_start': str(train_times.min().date()),
            'train_end': str(train_times.max().date()),
            'val_start': str(val_times.min().date()),
            'val_end': str(val_times.max().date()),
        })
    return folds


def sample_random_ranker_params(rng: np.random.Generator) -> dict:
    return {
        'learning_rate': float(np.exp(rng.uniform(np.log(0.02), np.log(0.10)))),
        'max_depth': int(rng.integers(4, 11)),
        'min_child_weight': float(np.exp(rng.uniform(np.log(1.0), np.log(50.0)))),
        'subsample': float(rng.uniform(0.60, 1.00)),
        'colsample_bytree': float(rng.uniform(0.50, 1.00)),
        'gamma': float(rng.uniform(0.0, 5.0)),
        'reg_alpha': float(np.exp(rng.uniform(np.log(1e-4), np.log(10.0)))),
        'reg_lambda': float(np.exp(rng.uniform(np.log(0.1), np.log(20.0)))),
    }


def tune_ranker_params(train_df: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series, top_k: int = 5):
    time_folds = make_temporal_folds(train_df, time_blocks=STAGE2_TIME_BLOCKS)
    rng = np.random.default_rng(RANDOM_STATE)
    trial_records = []
    fold_records = []

    def evaluate_param_set(params: dict, trial_label: str) -> dict:
        fold_summaries = []
        for fold in time_folds:
            fold_train = train_df[train_df['track_id'].isin(fold['train_track_ids'])].copy()
            fold_val = train_df[train_df['track_id'].isin(fold['val_track_ids'])].copy()
            fold_fill_values = fold_train[feature_cols].median(numeric_only=True)
            _, X_train_fold, y_train_fold, group_train_fold = prepare_ranker_inputs(fold_train, feature_cols, fold_fill_values)
            _, X_val_fold, y_val_fold, group_val_fold = prepare_ranker_inputs(fold_val, feature_cols, fold_fill_values)

            model = make_ranker(params)
            model.fit(
                X_train_fold,
                y_train_fold,
                group=group_train_fold,
                eval_set=[(X_val_fold, y_val_fold)],
                eval_group=[group_val_fold],
                verbose=False,
            )
            scored_val = score_ranker(model, fold_val, feature_cols, fold_fill_values, model_name='xgb_ranker_fold')
            summary, _ = evaluate_ranked_candidates(scored_val, k=top_k)
            best_iteration = getattr(model, 'best_iteration', None)
            resolved_rounds = int(best_iteration + 1) if best_iteration is not None else STAGE2_TUNING_N_ESTIMATORS
            record = {
                'trial_label': trial_label,
                'fold': fold['fold'],
                'best_iteration': resolved_rounds,
                'roc_auc': summary['binary']['roc_auc'],
                'average_precision': summary['binary']['average_precision'],
                f'precision@{top_k}': summary['ranking_all_tracks'][f'precision@{top_k}'],
                f'recall@{top_k}': summary['ranking_all_tracks'][f'recall@{top_k}'],
                f'hit_rate@{top_k}': summary['ranking_all_tracks'][f'hit_rate@{top_k}'],
                f'ndcg@{top_k}': summary['ranking_all_tracks'][f'ndcg@{top_k}'],
                f'map@{top_k}': summary['ranking_all_tracks'][f'map@{top_k}'],
            }
            fold_summaries.append(record)
            fold_records.append(record | params)

        fold_local_df = pd.DataFrame(fold_summaries)
        return {
            'trial_label': trial_label,
            'mean_best_iteration': float(fold_local_df['best_iteration'].mean()),
            'mean_roc_auc': float(fold_local_df['roc_auc'].mean()),
            'mean_average_precision': float(fold_local_df['average_precision'].mean()),
            f'mean_precision@{top_k}': float(fold_local_df[f'precision@{top_k}'].mean()),
            f'mean_recall@{top_k}': float(fold_local_df[f'recall@{top_k}'].mean()),
            f'mean_hit_rate@{top_k}': float(fold_local_df[f'hit_rate@{top_k}'].mean()),
            f'mean_ndcg@{top_k}': float(fold_local_df[f'ndcg@{top_k}'].mean()),
            f'mean_map@{top_k}': float(fold_local_df[f'map@{top_k}'].mean()),
            **params,
        }

    for trial_idx in range(STAGE2_TUNING_TRIALS):
        sampled_params = sample_random_ranker_params(rng)
        trial_records.append(evaluate_param_set(sampled_params, trial_label=f'random_{trial_idx:03d}'))

    trial_df = pd.DataFrame(trial_records).sort_values(f'mean_ndcg@{top_k}', ascending=False).reset_index(drop=True)
    fold_df = pd.DataFrame(fold_records)
    best_row = trial_df.iloc[0].to_dict()
    best_params = {key: best_row[key] for key in STAGE2_FALLBACK_PARAMS}
    final_n_estimators = int(np.ceil(best_row['mean_best_iteration'] * STAGE2_FINAL_N_ESTIMATOR_BUFFER))
    return best_params, final_n_estimators, trial_df, fold_df


def fit_ranker_with_validation(train_part: pd.DataFrame, val_part: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series, params: dict):
    _, X_train_part, y_train_part, group_train_part = prepare_ranker_inputs(train_part, feature_cols, fill_values)
    _, X_val_part, y_val_part, group_val_part = prepare_ranker_inputs(val_part, feature_cols, fill_values)
    model = make_ranker(params)
    model.fit(
        X_train_part,
        y_train_part,
        group=group_train_part,
        eval_set=[(X_val_part, y_val_part)],
        eval_group=[group_val_part],
        verbose=False,
    )
    return model


def fit_final_ranker(train_part: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series, params: dict, n_estimators: int):
    _, X_train_part, y_train_part, group_train_part = prepare_ranker_inputs(train_part, feature_cols, fill_values)
    model = make_ranker(params, n_estimators=n_estimators, early_stopping_rounds=None)
    model.fit(X_train_part, y_train_part, group=group_train_part, verbose=False)
    return model



def transform_target(y: pd.Series | np.ndarray, target_transform: str) -> np.ndarray:
    y_arr = np.asarray(y, dtype=float)
    if target_transform == 'log1p':
        return np.log1p(y_arr)
    return y_arr


def inverse_transform_target(y: np.ndarray, target_transform: str) -> np.ndarray:
    y_arr = np.asarray(y, dtype=float)
    if target_transform == 'log1p':
        return np.expm1(y_arr)
    return y_arr


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    abs_err = np.abs(y_true - y_pred)
    return {
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'median_ae': float(median_absolute_error(y_true, y_pred)),
        'pct_within_3_days': float(np.mean(abs_err <= 3.0)),
        'pct_within_7_days': float(np.mean(abs_err <= 7.0)),
    }


def make_regressor(params: dict, n_estimators: int = STAGE3_N_ESTIMATORS, early_stopping_rounds: int | None = STAGE3_EARLY_STOPPING_ROUNDS):
    init_kwargs = {
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'tree_method': 'hist',
        'n_estimators': n_estimators,
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        **params,
    }
    if early_stopping_rounds is not None:
        init_kwargs['early_stopping_rounds'] = early_stopping_rounds
    return xgb.XGBRegressor(**init_kwargs)


def fit_regressor_with_validation(train_pos: pd.DataFrame, val_pos: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series, params: dict, target_transform: str = 'identity'):
    X_train = make_feature_matrix(train_pos, feature_cols, fill_values)
    y_train = transform_target(train_pos['days_to_entry'].astype(float), target_transform)
    X_val = make_feature_matrix(val_pos, feature_cols, fill_values)
    y_val = transform_target(val_pos['days_to_entry'].astype(float), target_transform)
    model = make_regressor(params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return model


def fit_final_regressor(train_pos: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series, params: dict, n_estimators: int, target_transform: str = 'identity'):
    X_train = make_feature_matrix(train_pos, feature_cols, fill_values)
    y_train = transform_target(train_pos['days_to_entry'].astype(float), target_transform)
    model = make_regressor(params, n_estimators=n_estimators, early_stopping_rounds=None)
    model.fit(X_train, y_train, verbose=False)
    return model


def choose_regressor_params(train_pos: pd.DataFrame, val_pos: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series, param_grid: list[dict] | None = None, transforms: list[str] | None = None):
    trial_rows = []
    models = {}
    param_grid = STAGE3_PARAM_GRID if param_grid is None else param_grid
    transforms = STAGE3_TARGET_TRANSFORMS if transforms is None else transforms
    for idx, params in enumerate(param_grid):
        for target_transform in transforms:
            label = f'regressor_{idx:02d}_{target_transform}'
            model = fit_regressor_with_validation(train_pos, val_pos, feature_cols, fill_values, params, target_transform=target_transform)
            raw_preds = model.predict(make_feature_matrix(val_pos, feature_cols, fill_values))
            preds = np.clip(inverse_transform_target(raw_preds, target_transform), 1.0, 60.0)
            metrics = regression_metrics(val_pos['days_to_entry'].astype(float).to_numpy(), preds)
            best_iteration = getattr(model, 'best_iteration', None)
            trial_rows.append({
                'trial_label': label,
                'target_transform': target_transform,
                'best_iteration': None if best_iteration is None else int(best_iteration + 1),
                **params,
                **metrics,
            })
            models[label] = model
    trial_df = pd.DataFrame(trial_rows).sort_values(['mae', 'rmse', 'median_ae'], ascending=[True, True, True]).reset_index(drop=True)
    best_row = trial_df.iloc[0].to_dict()
    best_model = models[best_row['trial_label']]
    best_params = {key: best_row[key] for key in STAGE3_DEFAULT_PARAMS}
    best_transform = best_row['target_transform']
    return best_params, best_transform, best_model, trial_df


def score_regressor(model, df: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series, target_transform: str = 'identity') -> pd.DataFrame:
    preds = model.predict(make_feature_matrix(df, feature_cols, fill_values))
    preds = inverse_transform_target(preds, target_transform)
    scored = df[['track_id', 'observation_time', 'target_country', 'did_enter_within_60d', 'days_to_entry']].copy()
    scored['predicted_days_to_entry'] = np.clip(preds, 1.0, 60.0)
    return scored


def add_stage1_to_row_predictions(row_scored_df: pd.DataFrame, stage1_scored_df: pd.DataFrame, threshold_map: dict) -> pd.DataFrame:
    merged = row_scored_df.merge(
        stage1_scored_df[['track_id', 'score']].rename(columns={'score': 'spread_score'}),
        on='track_id',
        how='left',
    )
    for floor, threshold in threshold_map.items():
        col = f'spread_decision_floor_{float(floor):.2f}'.replace('.', '_')
        merged[col] = (merged['spread_score'] >= float(threshold)).astype(int)
    return merged


def add_regression_predictions(row_scored_df: pd.DataFrame, reg_scored_df: pd.DataFrame) -> pd.DataFrame:
    return row_scored_df.merge(
        reg_scored_df[['track_id', 'target_country', 'predicted_days_to_entry']],
        on=['track_id', 'target_country'],
        how='left',
    )


def add_predicted_rank(scored_df: pd.DataFrame) -> pd.DataFrame:
    ranked = scored_df.sort_values(['track_id', 'score', 'target_new_entry_rate_30d'], ascending=[True, False, False]).copy()
    ranked['predicted_rank'] = ranked.groupby('track_id').cumcount().add(1).astype(int)
    return ranked


def build_topk_recommendations(scored_df: pd.DataFrame, gate_col: str, k: int = 5) -> pd.DataFrame:
    ranked = scored_df.sort_values(['track_id', 'score', 'target_new_entry_rate_30d'], ascending=[True, False, False]).copy()
    flagged = ranked[ranked[gate_col] == 1].copy()
    topk = flagged.groupby('track_id', sort=False).head(k).copy()
    topk['recommended_in_top_k'] = 1
    return topk


def evaluate_gated_pipeline(scored_df: pd.DataFrame, gate_col: str, k: int = 5) -> tuple[dict, pd.DataFrame, dict, pd.DataFrame]:
    rows = []
    hit_rows = []
    topk_rows = []
    for track_id, group in scored_df.groupby('track_id', sort=False):
        group = group.sort_values(['score', 'target_new_entry_rate_30d'], ascending=[False, False]).reset_index(drop=True)
        labels = group['did_enter_within_60d'].to_numpy(dtype=int)
        positives = int(labels.sum())
        gated = bool(group[gate_col].iloc[0]) if not group.empty else False
        top = group.head(k).copy() if gated else group.head(0).copy()
        top_labels = top['did_enter_within_60d'].to_numpy(dtype=int)
        hits = int(top_labels.sum())

        precision = hits / k
        recall = hits / positives if positives else np.nan
        hit_rate = float(hits > 0) if positives else np.nan

        discounts = np.log2(np.arange(2, len(top_labels) + 2)) if len(top_labels) else np.array([])
        dcg = float(((2 ** top_labels - 1) / discounts).sum()) if len(top_labels) else 0.0
        ideal = np.sort(labels)[::-1][:k]
        idcg = float(((2 ** ideal - 1) / np.log2(np.arange(2, len(ideal) + 2))).sum()) if positives else 0.0
        ndcg = dcg / idcg if idcg > 0 else np.nan

        ap_accum = 0.0
        running_hits = 0
        for rank, rel in enumerate(top_labels, start=1):
            if rel:
                running_hits += 1
                ap_accum += running_hits / rank
        map_k = ap_accum / min(positives, k) if positives else np.nan

        rows.append({
            'track_id': track_id,
            'positives': positives,
            'stage1_flagged': int(gated),
            'top_k_hits': hits,
            f'precision@{k}': precision,
            f'recall@{k}': recall,
            f'hit_rate@{k}': hit_rate,
            f'ndcg@{k}': ndcg,
            f'map@{k}': map_k,
        })

        if not top.empty:
            topk_rows.append(top)
            hits_df = top[top['did_enter_within_60d'] == 1].copy()
            if not hits_df.empty:
                hit_rows.append(hits_df)

    metric_df = pd.DataFrame(rows)
    positive_mask = metric_df['positives'] > 0
    summary = {
        'tracks': int(metric_df.shape[0]),
        'positive_tracks': int(positive_mask.sum()),
        'flagged_tracks': int(metric_df['stage1_flagged'].sum()),
        f'precision@{k}': float(metric_df[f'precision@{k}'].mean()),
        f'recall@{k}': float(metric_df.loc[positive_mask, f'recall@{k}'].mean()) if positive_mask.any() else None,
        f'hit_rate@{k}': float(metric_df.loc[positive_mask, f'hit_rate@{k}'].mean()) if positive_mask.any() else None,
        f'ndcg@{k}': float(metric_df.loc[positive_mask, f'ndcg@{k}'].mean()) if positive_mask.any() else None,
        f'map@{k}': float(metric_df.loc[positive_mask, f'map@{k}'].mean()) if positive_mask.any() else None,
    }
    topk_df = pd.concat(topk_rows, ignore_index=True) if topk_rows else scored_df.head(0).copy()
    hit_df = pd.concat(hit_rows, ignore_index=True) if hit_rows else scored_df.head(0).copy()
    if not hit_df.empty:
        day_metrics = regression_metrics(hit_df['days_to_entry'].astype(float).to_numpy(), hit_df['predicted_days_to_entry'].astype(float).to_numpy())
        day_metrics['evaluated_rows'] = int(len(hit_df))
    else:
        day_metrics = {
            'mae': None,
            'rmse': None,
            'median_ae': None,
            'pct_within_3_days': None,
            'pct_within_7_days': None,
            'evaluated_rows': 0,
        }
    return summary, metric_df, day_metrics, topk_df


def maybe_compute_shap(model, df: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series, model_name: str) -> pd.DataFrame | None:
    if not RUN_SHAP:
        print('Skipping SHAP section.')
        return None
    try:
        import shap
    except ImportError:
        print('SHAP is not installed. Run `pip install -r requirements.txt` and re-run this section.')
        return None

    shap_track_limit = min(SHAP_SAMPLE_TRACKS, df['track_id'].nunique())
    shap_track_ids = (
        df[['track_id']]
        .drop_duplicates()
        .sort_values('track_id')
        .head(shap_track_limit)['track_id']
        .tolist()
    )
    shap_df = df[df['track_id'].isin(shap_track_ids)].copy()
    X_shap = make_feature_matrix(shap_df, feature_cols, fill_values)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_shap)
    if isinstance(shap_values, list):
        shap_values = shap_values[-1]
    summary_df = pd.DataFrame({
        'feature': feature_cols,
        'mean_abs_shap': np.abs(shap_values).mean(axis=0),
        'category': [feature_category(name) for name in feature_cols],
        'model_name': model_name,
    }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
    display(summary_df.head(20))
    return summary_df


In [ ]:
row_train_df = load_row_level_split(TRAIN_PATH, TRAIN_MAX_TRACKS)
row_val_df = load_row_level_split(VAL_PATH, VAL_MAX_TRACKS)
row_test_df = load_row_level_split(TEST_PATH, TEST_MAX_TRACKS)

track_train_df = load_track_level_split(TRAIN_PATH, TRAIN_MAX_TRACKS)
track_val_df = load_track_level_split(VAL_PATH, VAL_MAX_TRACKS)
track_test_df = load_track_level_split(TEST_PATH, TEST_MAX_TRACKS)

row_feature_cols = [col for col in row_train_df.columns if col not in FEATURE_EXCLUDE]
row_fill_values_train = row_train_df[row_feature_cols].median(numeric_only=True)
row_fill_values_final = pd.concat([row_train_df, row_val_df], ignore_index=True)[row_feature_cols].median(numeric_only=True)

track_feature_exclude = ['track_id', 'observation_time', 'will_spread', 'min_days_to_spread']
track_feature_cols = [col for col in track_train_df.columns if col not in track_feature_exclude]
track_fill_values = track_train_df[track_feature_cols].median(numeric_only=True)

print('Row-level dataset:')
print(f"Train rows: {len(row_train_df):,} | tracks: {row_train_df['track_id'].nunique():,}")
print(f"Val rows:   {len(row_val_df):,} | tracks: {row_val_df['track_id'].nunique():,}")
print(f"Test rows:  {len(row_test_df):,} | tracks: {row_test_df['track_id'].nunique():,}")
print(f"Row feature count: {len(row_feature_cols)}")
print()
print('Track-level dataset:')
print(f"Train tracks: {len(track_train_df):,} | positives: {int(track_train_df['will_spread'].sum()):,} ({track_train_df['will_spread'].mean() * 100:.2f}%)")
print(f"Val tracks:   {len(track_val_df):,} | positives: {int(track_val_df['will_spread'].sum()):,} ({track_val_df['will_spread'].mean() * 100:.2f}%)")
print(f"Test tracks:  {len(track_test_df):,} | positives: {int(track_test_df['will_spread'].sum()):,} ({track_test_df['will_spread'].mean() * 100:.2f}%)")
print(f"Track feature count: {len(track_feature_cols)}")

positive_train_rows = row_train_df[(row_train_df['did_enter_within_60d'] == 1) & row_train_df['days_to_entry'].notna()].copy()
positive_val_rows = row_val_df[(row_val_df['did_enter_within_60d'] == 1) & row_val_df['days_to_entry'].notna()].copy()
positive_test_rows = row_test_df[(row_test_df['did_enter_within_60d'] == 1) & row_test_df['days_to_entry'].notna()].copy()
print()
print('Positive regression rows:')
print(f'Train positives: {len(positive_train_rows):,}')
print(f'Val positives:   {len(positive_val_rows):,}')
print(f'Test positives:  {len(positive_test_rows):,}')


In [ ]:
X_track_train = make_feature_matrix(track_train_df, track_feature_cols, track_fill_values)
X_track_val = make_feature_matrix(track_val_df, track_feature_cols, track_fill_values)
X_track_test = make_feature_matrix(track_test_df, track_feature_cols, track_fill_values)
y_track_train = track_train_df['will_spread'].astype(int)
y_track_val = track_val_df['will_spread'].astype(int)
y_track_test = track_test_df['will_spread'].astype(int)

stage1_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric=['aucpr'],
    tree_method='hist',
    n_estimators=STAGE1_N_ESTIMATORS,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    reg_alpha=0.0,
    max_delta_step=1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    early_stopping_rounds=STAGE1_EARLY_STOPPING_ROUNDS,
    **STAGE1_FIXED_PARAMS,
)
stage1_model.fit(X_track_train, y_track_train, eval_set=[(X_track_val, y_track_val)], verbose=False)

stage1_val_scores = stage1_model.predict_proba(X_track_val)[:, 1]
stage1_test_scores = stage1_model.predict_proba(X_track_test)[:, 1]

stage1_val_scored = track_val_df[['track_id', 'observation_time', 'origin_country_count_at_obs', 'candidate_count', 'will_spread']].copy()
stage1_val_scored['score'] = stage1_val_scores
stage1_test_scored = track_test_df[['track_id', 'observation_time', 'origin_country_count_at_obs', 'candidate_count', 'will_spread']].copy()
stage1_test_scored['score'] = stage1_test_scores

stage1_val_thresholds, stage1_threshold_map, stage1_threshold_sweep = build_business_threshold_summary(
    y_track_val,
    stage1_val_scores,
    BUSINESS_PRECISION_FLOORS,
    beta=2.0,
)

stage1_threshold_rows = []
for floor in BUSINESS_PRECISION_FLOORS:
    threshold = float(stage1_threshold_map[float(floor)])
    selection_reason = stage1_val_thresholds.loc[stage1_val_thresholds['precision_floor'] == float(floor), 'selection_reason'].iloc[0]
    stage1_threshold_rows.append({
        'split': 'validation',
        'precision_floor': float(floor),
        'threshold': threshold,
        'selection_reason': selection_reason,
        **binary_summary(y_track_val, stage1_val_scores, threshold=threshold, beta=2.0),
    })
    stage1_threshold_rows.append({
        'split': 'test',
        'precision_floor': float(floor),
        'threshold': threshold,
        'selection_reason': selection_reason + ' (chosen on validation)',
        **binary_summary(y_track_test, stage1_test_scores, threshold=threshold, beta=2.0),
    })
stage1_threshold_df = pd.DataFrame(stage1_threshold_rows)
print('Stage 1 threshold summary:')
display(stage1_threshold_df)

for floor, threshold in stage1_threshold_map.items():
    col = f'spread_decision_floor_{float(floor):.2f}'.replace('.', '_')
    stage1_val_scored[col] = (stage1_val_scored['score'] >= float(threshold)).astype(int)
    stage1_test_scored[col] = (stage1_test_scored['score'] >= float(threshold)).astype(int)

stage1_handoff_rows = []
for split_name, scored_df in [('validation', stage1_val_scored), ('test', stage1_test_scored)]:
    for floor in BUSINESS_PRECISION_FLOORS:
        pred_col = f'spread_decision_floor_{float(floor):.2f}'.replace('.', '_')
        flagged = scored_df[pred_col] == 1
        stage1_handoff_rows.append({
            'split': split_name,
            'precision_floor': float(floor),
            'tracks_total': int(len(scored_df)),
            'tracks_flagged_to_ranker': int(flagged.sum()),
            'flagged_share': float(flagged.mean()),
            'flagged_per_1000_tracks': float(flagged.mean() * 1000.0),
            'true_spreaders_captured': int(scored_df.loc[flagged, 'will_spread'].sum()),
            'missed_spreaders': int(scored_df.loc[~flagged, 'will_spread'].sum()),
            'false_positives_sent_to_ranker': int(((flagged) & (scored_df['will_spread'] == 0)).sum()),
        })
stage1_handoff_df = pd.DataFrame(stage1_handoff_rows)
print()
print('Stage 1 handoff summary:')
display(stage1_handoff_df)

stage1_base_threshold_df = stage1_threshold_df.copy()
stage1_base_threshold_map = {float(key): float(value) for key, value in stage1_threshold_map.items()}
stage1_base_threshold_sweep = stage1_threshold_sweep.copy()
stage1_base_handoff_df = stage1_handoff_df.copy()

stage1_summary = {
    'validation': {
        'thresholds': stage1_threshold_df[stage1_threshold_df['split'] == 'validation'].to_dict(orient='records'),
    },
    'test': {
        'thresholds': stage1_threshold_df[stage1_threshold_df['split'] == 'test'].to_dict(orient='records'),
    },
    'selected_params': STAGE1_FIXED_PARAMS,
    'best_iteration': None if getattr(stage1_model, 'best_iteration', None) is None else int(stage1_model.best_iteration),
}


In [ ]:
frozen_ranker_params, frozen_final_n_estimators = load_frozen_ranker_config(RANKER_SUMMARY_PATH)
if RUN_RANKER_RETUNING:
    stage2_params, stage2_final_n_estimators, stage2_tuning_df, stage2_fold_df = tune_ranker_params(
        row_train_df,
        row_feature_cols,
        row_fill_values_train,
        top_k=TOP_K,
    )
    stage2_tuning_method = 'random_search'
else:
    stage2_params = frozen_ranker_params
    stage2_final_n_estimators = frozen_final_n_estimators
    stage2_tuning_df = pd.DataFrame([{**stage2_params, 'source': 'frozen_best_ranker_params'}])
    stage2_fold_df = pd.DataFrame()
    stage2_tuning_method = 'frozen_params'

stage2_val_model = fit_ranker_with_validation(row_train_df, row_val_df, row_feature_cols, row_fill_values_train, stage2_params)
stage2_val_best_iteration = getattr(stage2_val_model, 'best_iteration', None)
stage2_val_best_rounds = int(stage2_val_best_iteration + 1) if stage2_val_best_iteration is not None else stage2_final_n_estimators

stage2_val_predictions = {
    'xgb_ranker_validation': score_ranker(stage2_val_model, row_val_df, row_feature_cols, row_fill_values_train, model_name='xgb_ranker_validation'),
    'naive_popularity_baseline': score_naive_popularity(row_val_df),
}
stage2_val_results = {}
stage2_val_track_tables = {}
for model_name, scored_df in stage2_val_predictions.items():
    summary, track_metrics = evaluate_ranked_candidates(scored_df, k=TOP_K)
    stage2_val_results[model_name] = summary
    stage2_val_track_tables[model_name] = track_metrics

combined_train_val_df = pd.concat([row_train_df, row_val_df], ignore_index=True)
if RUN_RANKER_RETUNING:
    stage2_final_n_estimators = int(np.ceil(max(stage2_final_n_estimators, stage2_val_best_rounds) * STAGE2_FINAL_N_ESTIMATOR_BUFFER))
else:
    stage2_final_n_estimators = int(max(stage2_final_n_estimators, stage2_val_best_rounds))
stage2_final_model = fit_final_ranker(combined_train_val_df, row_feature_cols, row_fill_values_final, stage2_params, n_estimators=stage2_final_n_estimators)

stage2_test_predictions = {
    'xgb_ranker_final': score_ranker(stage2_final_model, row_test_df, row_feature_cols, row_fill_values_final, model_name='xgb_ranker_final'),
    'naive_popularity_baseline': score_naive_popularity(row_test_df),
}
stage2_test_results = {}
stage2_test_track_tables = {}
for model_name, scored_df in stage2_test_predictions.items():
    summary, track_metrics = evaluate_ranked_candidates(scored_df, k=TOP_K)
    stage2_test_results[model_name] = summary
    stage2_test_track_tables[model_name] = track_metrics

stage2_comparison_rows = []
for split_name, result_map in [('validation', stage2_val_results), ('test', stage2_test_results)]:
    for model_name, summary in result_map.items():
        stage2_comparison_rows.append({
            'split': split_name,
            'model': model_name,
            'roc_auc': summary['binary']['roc_auc'],
            'average_precision': summary['binary']['average_precision'],
            f'precision@{TOP_K}': summary['ranking_all_tracks'][f'precision@{TOP_K}'],
            f'recall@{TOP_K}': summary['ranking_all_tracks'][f'recall@{TOP_K}'],
            f'hit_rate@{TOP_K}': summary['ranking_all_tracks'][f'hit_rate@{TOP_K}'],
            f'ndcg@{TOP_K}': summary['ranking_all_tracks'][f'ndcg@{TOP_K}'],
            f'map@{TOP_K}': summary['ranking_all_tracks'][f'map@{TOP_K}'],
            f'positive_recall@{TOP_K}': summary['ranking_positive_tracks'][f'recall@{TOP_K}'],
            f'positive_hit_rate@{TOP_K}': summary['ranking_positive_tracks'][f'hit_rate@{TOP_K}'],
            f'positive_ndcg@{TOP_K}': summary['ranking_positive_tracks'][f'ndcg@{TOP_K}'],
            f'positive_map@{TOP_K}': summary['ranking_positive_tracks'][f'map@{TOP_K}'],
            'tracks': summary['candidate_stats']['tracks'],
            'positive_tracks': summary['candidate_stats']['positive_tracks'],
            'avg_candidates_per_track': summary['candidate_stats']['avg_candidates_per_track'],
        })
stage2_comparison_df = pd.DataFrame(stage2_comparison_rows).sort_values(['split', 'model']).reset_index(drop=True)
print('Stage 2 ranking comparison:')
display(stage2_comparison_df)


pipeline_threshold_rows = []
pipeline_selected_df, pipeline_threshold_map, pipeline_candidate_df = choose_pipeline_thresholds(
    stage1_val_scored,
    stage2_val_track_tables['xgb_ranker_validation'],
    BUSINESS_PRECISION_FLOORS,
    k=TOP_K,
)

for policy_name, threshold_map_source, selection_df in [
    ('stage1_recall_floor', stage1_base_threshold_map, stage1_base_threshold_df[['split', 'precision_floor', 'selection_reason']].copy()),
    ('pipeline_recall_at_k', pipeline_threshold_map, pipeline_selected_df[['precision_floor', 'selection_reason']].copy()),
]:
    for split_name, stage1_scored_df_local, y_true_local, ranker_track_df in [
        ('validation', stage1_val_scored, y_track_val, stage2_val_track_tables['xgb_ranker_validation']),
        ('test', stage1_test_scored, y_track_test, stage2_test_track_tables['xgb_ranker_final']),
    ]:
        for floor in BUSINESS_PRECISION_FLOORS:
            threshold = float(threshold_map_source[float(floor)])
            if policy_name == 'stage1_recall_floor':
                selection_reason = selection_df[(selection_df['split'] == split_name) & (selection_df['precision_floor'] == float(floor))]['selection_reason'].iloc[0]
            else:
                selection_reason = selection_df[selection_df['precision_floor'] == float(floor)]['selection_reason'].iloc[0]
                if split_name == 'test':
                    selection_reason = selection_reason + ' (chosen on validation)'
            binary_row = binary_summary(y_true_local, stage1_scored_df_local['score'].to_numpy(), threshold=threshold, beta=2.0)
            pipeline_row = summarize_gated_track_metrics(stage1_scored_df_local, ranker_track_df, threshold=threshold, k=TOP_K)
            pipeline_threshold_rows.append({
                'policy': policy_name,
                'split': split_name,
                'precision_floor': float(floor),
                'threshold': threshold,
                'selection_reason': selection_reason,
                **binary_row,
                **pipeline_row,
            })

stage1_threshold_df = pd.DataFrame(pipeline_threshold_rows).sort_values(['policy', 'split', 'precision_floor']).reset_index(drop=True)
print()
print('Stage 1 policy comparison:')
display(stage1_threshold_df)

if USE_PIPELINE_AWARE_THRESHOLDS:
    stage1_threshold_map = {float(key): float(value) for key, value in pipeline_threshold_map.items()}
    stage1_threshold_selection_policy = 'pipeline_recall_at_k'
    stage1_threshold_sweep = pipeline_candidate_df.copy()
else:
    stage1_threshold_map = stage1_base_threshold_map.copy()
    stage1_threshold_selection_policy = 'stage1_recall_floor'
    stage1_threshold_sweep = stage1_base_threshold_sweep.copy()

for floor, threshold in stage1_threshold_map.items():
    col = f'spread_decision_floor_{float(floor):.2f}'.replace('.', '_')
    stage1_val_scored[col] = (stage1_val_scored['score'] >= float(threshold)).astype(int)
    stage1_test_scored[col] = (stage1_test_scored['score'] >= float(threshold)).astype(int)

stage1_handoff_rows = []
for split_name, scored_df in [('validation', stage1_val_scored), ('test', stage1_test_scored)]:
    for floor in BUSINESS_PRECISION_FLOORS:
        pred_col = f'spread_decision_floor_{float(floor):.2f}'.replace('.', '_')
        flagged = scored_df[pred_col] == 1
        stage1_handoff_rows.append({
            'policy': stage1_threshold_selection_policy,
            'split': split_name,
            'precision_floor': float(floor),
            'tracks_total': int(len(scored_df)),
            'tracks_flagged_to_ranker': int(flagged.sum()),
            'flagged_share': float(flagged.mean()),
            'flagged_per_1000_tracks': float(flagged.mean() * 1000.0),
            'true_spreaders_captured': int(scored_df.loc[flagged, 'will_spread'].sum()),
            'missed_spreaders': int(scored_df.loc[~flagged, 'will_spread'].sum()),
            'false_positives_sent_to_ranker': int(((flagged) & (scored_df['will_spread'] == 0)).sum()),
        })
stage1_handoff_df = pd.DataFrame(stage1_handoff_rows).sort_values(['split', 'precision_floor']).reset_index(drop=True)
print()
print(f'Active stage 1 threshold policy: {stage1_threshold_selection_policy}')
print('Stage 1 active handoff summary:')
display(stage1_handoff_df)

stage1_summary = {
    'active_threshold_policy': stage1_threshold_selection_policy,
    'validation': {
        'thresholds': stage1_threshold_df[stage1_threshold_df['split'] == 'validation'].to_dict(orient='records'),
    },
    'test': {
        'thresholds': stage1_threshold_df[stage1_threshold_df['split'] == 'test'].to_dict(orient='records'),
    },
    'selected_params': STAGE1_FIXED_PARAMS,
    'best_iteration': None if getattr(stage1_model, 'best_iteration', None) is None else int(stage1_model.best_iteration),
}

stage2_summary = {
    'tuning_method': stage2_tuning_method,
    'params': stage2_params,
    'final_n_estimators': int(stage2_final_n_estimators),
    'validation': stage2_val_results,
    'test': stage2_test_results,
}


In [ ]:
if RUN_REGRESSOR_TUNING:
    stage3_params, stage3_target_transform, stage3_val_model, stage3_tuning_df = choose_regressor_params(
        positive_train_rows,
        positive_val_rows,
        row_feature_cols,
        row_fill_values_train,
    )
    stage3_tuning_method = 'grid_search'
else:
    stage3_params, stage3_target_transform, stage3_val_model, stage3_tuning_df = choose_regressor_params(
        positive_train_rows,
        positive_val_rows,
        row_feature_cols,
        row_fill_values_train,
        param_grid=[STAGE3_DEFAULT_PARAMS.copy()],
        transforms=STAGE3_TARGET_TRANSFORMS,
    )
    stage3_tuning_method = 'transform_selection'

stage3_val_best_iteration = getattr(stage3_val_model, 'best_iteration', None)
stage3_val_best_rounds = int(stage3_val_best_iteration + 1) if stage3_val_best_iteration is not None else STAGE3_N_ESTIMATORS

stage3_val_positive_scored = score_regressor(stage3_val_model, positive_val_rows, row_feature_cols, row_fill_values_train, target_transform=stage3_target_transform)
stage3_val_metrics = regression_metrics(
    stage3_val_positive_scored['days_to_entry'].astype(float).to_numpy(),
    stage3_val_positive_scored['predicted_days_to_entry'].astype(float).to_numpy(),
)
stage3_val_metrics['rows'] = int(len(stage3_val_positive_scored))

combined_positive_train_val = pd.concat([positive_train_rows, positive_val_rows], ignore_index=True)
stage3_final_n_estimators = int(np.ceil(stage3_val_best_rounds * 1.10))
stage3_final_model = fit_final_regressor(
    combined_positive_train_val,
    row_feature_cols,
    row_fill_values_final,
    stage3_params,
    n_estimators=stage3_final_n_estimators,
    target_transform=stage3_target_transform,
)

stage3_test_positive_scored = score_regressor(stage3_final_model, positive_test_rows, row_feature_cols, row_fill_values_final, target_transform=stage3_target_transform)
stage3_test_metrics = regression_metrics(
    stage3_test_positive_scored['days_to_entry'].astype(float).to_numpy(),
    stage3_test_positive_scored['predicted_days_to_entry'].astype(float).to_numpy(),
)
stage3_test_metrics['rows'] = int(len(stage3_test_positive_scored))

stage3_val_all_scored = score_regressor(stage3_val_model, row_val_df, row_feature_cols, row_fill_values_train, target_transform=stage3_target_transform)
stage3_test_all_scored = score_regressor(stage3_final_model, row_test_df, row_feature_cols, row_fill_values_final, target_transform=stage3_target_transform)

stage3_summary_df = pd.DataFrame([
    {'split': 'validation', 'target_transform': stage3_target_transform, **stage3_val_metrics},
    {'split': 'test', 'target_transform': stage3_target_transform, **stage3_test_metrics},
]).reset_index(drop=True)
print('Stage 3 regression summary:')
display(stage3_summary_df)
print()
print('Stage 3 candidate comparison:')
display(stage3_tuning_df)

stage3_summary = {
    'tuning_method': stage3_tuning_method,
    'target_transform': stage3_target_transform,
    'params': stage3_params,
    'final_n_estimators': int(stage3_final_n_estimators),
    'validation': stage3_val_metrics,
    'test': stage3_test_metrics,
}


In [ ]:
val_pipeline_predictions = add_stage1_to_row_predictions(stage2_val_predictions['xgb_ranker_validation'], stage1_val_scored, stage1_threshold_map)
val_pipeline_predictions = add_regression_predictions(val_pipeline_predictions, stage3_val_all_scored)
val_pipeline_predictions = add_predicted_rank(val_pipeline_predictions)
val_pipeline_predictions = val_pipeline_predictions.rename(columns={'score': 'rank_score'})

test_pipeline_predictions = add_stage1_to_row_predictions(stage2_test_predictions['xgb_ranker_final'], stage1_test_scored, stage1_threshold_map)
test_pipeline_predictions = add_regression_predictions(test_pipeline_predictions, stage3_test_all_scored)
test_pipeline_predictions = add_predicted_rank(test_pipeline_predictions)
test_pipeline_predictions = test_pipeline_predictions.rename(columns={'score': 'rank_score'})

pipeline_rows = []
pipeline_track_tables = {}
pipeline_topk_tables = {}
for split_name, pipeline_df in [('validation', val_pipeline_predictions), ('test', test_pipeline_predictions)]:
    for floor in BUSINESS_PRECISION_FLOORS:
        gate_col = f'spread_decision_floor_{float(floor):.2f}'.replace('.', '_')
        gated_df = pipeline_df.copy().rename(columns={'rank_score': 'score'})
        ranking_summary, track_metrics, day_metrics, topk_df = evaluate_gated_pipeline(gated_df, gate_col=gate_col, k=TOP_K)
        topk_df = topk_df.rename(columns={'score': 'rank_score'})
        pipeline_rows.append({
            'split': split_name,
            'precision_floor': float(floor),
            **ranking_summary,
            **{f'days_{metric_name}': metric_value for metric_name, metric_value in day_metrics.items()},
        })
        pipeline_track_tables[(split_name, float(floor))] = track_metrics
        pipeline_topk_tables[(split_name, float(floor))] = topk_df

pipeline_summary_df = pd.DataFrame(pipeline_rows).sort_values(['split', 'precision_floor']).reset_index(drop=True)
print('End-to-end pipeline summary:')
display(pipeline_summary_df)

primary_gate_col = f'spread_decision_floor_{PRIMARY_PRECISION_FLOOR:.2f}'.replace('.', '_')
val_top5_recommendations = pipeline_topk_tables[('validation', float(PRIMARY_PRECISION_FLOOR))].copy()
test_top5_recommendations = pipeline_topk_tables[('test', float(PRIMARY_PRECISION_FLOOR))].copy()

selected_columns = [
    'track_id',
    'observation_time',
    'spread_score',
    primary_gate_col,
    'rank_score',
    'predicted_rank',
    'target_country',
    'predicted_days_to_entry',
    'did_enter_within_60d',
    'days_to_entry',
]
val_top5_recommendations = val_top5_recommendations[selected_columns].copy()
test_top5_recommendations = test_top5_recommendations[selected_columns].copy()

print()
print('Validation top-5 recommendations (primary business threshold):')
display(val_top5_recommendations.head(10))
print()
print('Test top-5 recommendations (primary business threshold):')
display(test_top5_recommendations.head(10))


In [ ]:
shap_summary_df = maybe_compute_shap(
    stage2_final_model,
    row_test_df,
    row_feature_cols,
    row_fill_values_final,
    model_name='stage2_ranker_final',
)

pipeline_summary = {
    'config': {
        'run_mode': RUN_MODE,
        'run_full_splits': RUN_FULL_SPLITS,
        'top_k': TOP_K,
        'primary_precision_floor': PRIMARY_PRECISION_FLOOR,
        'secondary_precision_floors': SECONDARY_PRECISION_FLOORS,
        'run_ranker_retuning': RUN_RANKER_RETUNING,
        'run_regressor_tuning': RUN_REGRESSOR_TUNING,
        'use_pipeline_aware_thresholds': USE_PIPELINE_AWARE_THRESHOLDS,
        'run_shap': RUN_SHAP,
        'shap_sample_tracks': SHAP_SAMPLE_TRACKS,
    },
    'data': {
        'train_path': str(TRAIN_PATH),
        'val_path': str(VAL_PATH),
        'test_path': str(TEST_PATH),
        'row_train_rows': int(len(row_train_df)),
        'row_val_rows': int(len(row_val_df)),
        'row_test_rows': int(len(row_test_df)),
        'track_train_tracks': int(len(track_train_df)),
        'track_val_tracks': int(len(track_val_df)),
        'track_test_tracks': int(len(track_test_df)),
        'positive_train_rows': int(len(positive_train_rows)),
        'positive_val_rows': int(len(positive_val_rows)),
        'positive_test_rows': int(len(positive_test_rows)),
    },
    'stage1': stage1_summary,
    'stage2': stage2_summary,
    'stage3': stage3_summary,
    'pipeline': {
        'summary_rows': pipeline_summary_df.to_dict(orient='records'),
    },
}

stage1_model_path = MODEL_DIR / 'stage1_will_spread_classifier.json'
stage2_model_path = MODEL_DIR / 'stage2_country_ranker.json'
stage3_model_path = MODEL_DIR / 'stage3_days_to_entry_regressor.json'
pipeline_summary_path = EVAL_DIR / 'pipeline_summary.json'
stage1_threshold_summary_path = EVAL_DIR / 'stage1_threshold_summary.parquet'
stage1_handoff_summary_path = EVAL_DIR / 'stage1_handoff_summary.parquet'
ranking_comparison_path = EVAL_DIR / 'ranking_comparison.parquet'
regression_summary_path = EVAL_DIR / 'regression_summary.parquet'
val_pipeline_predictions_path = EVAL_DIR / 'val_pipeline_predictions.parquet'
test_pipeline_predictions_path = EVAL_DIR / 'test_pipeline_predictions.parquet'
test_top5_recommendations_path = EVAL_DIR / 'test_top5_recommendations.parquet'
val_top5_recommendations_path = EVAL_DIR / 'val_top5_recommendations.parquet'
stage1_threshold_sweep_path = EVAL_DIR / 'stage1_threshold_sweep.parquet'
stage2_tuning_path = EVAL_DIR / 'stage2_tuning_results.parquet'
stage2_fold_path = EVAL_DIR / 'stage2_tuning_fold_metrics.parquet'
stage3_tuning_path = EVAL_DIR / 'stage3_tuning_results.parquet'
pipeline_topk_summary_path = EVAL_DIR / 'pipeline_topk_summary.parquet'
shap_summary_path = EVAL_DIR / 'shap_summary.parquet'

stage1_model.save_model(stage1_model_path)
stage2_final_model.save_model(stage2_model_path)
stage3_final_model.save_model(stage3_model_path)
with open(pipeline_summary_path, 'w', encoding='utf-8') as f:
    json.dump(pipeline_summary, f, indent=2)

con.register('stage1_threshold_df', stage1_threshold_df)
con.register('stage1_handoff_df', stage1_handoff_df)
con.register('stage2_comparison_df', stage2_comparison_df)
con.register('stage3_summary_df', stage3_summary_df)
con.register('val_pipeline_predictions_df', val_pipeline_predictions)
con.register('test_pipeline_predictions_df', test_pipeline_predictions)
con.register('val_top5_recommendations_df', val_top5_recommendations)
con.register('test_top5_recommendations_df', test_top5_recommendations)
con.register('stage1_threshold_sweep_df', stage1_threshold_sweep)
con.register('stage2_tuning_df', stage2_tuning_df)
con.register('pipeline_summary_df', pipeline_summary_df)

con.execute(f"COPY stage1_threshold_df TO '{stage1_threshold_summary_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY stage1_handoff_df TO '{stage1_handoff_summary_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY stage2_comparison_df TO '{ranking_comparison_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY stage3_summary_df TO '{regression_summary_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY val_pipeline_predictions_df TO '{val_pipeline_predictions_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY test_pipeline_predictions_df TO '{test_pipeline_predictions_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY val_top5_recommendations_df TO '{val_top5_recommendations_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY test_top5_recommendations_df TO '{test_top5_recommendations_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY stage1_threshold_sweep_df TO '{stage1_threshold_sweep_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY stage2_tuning_df TO '{stage2_tuning_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY pipeline_summary_df TO '{pipeline_topk_summary_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")

if not stage2_fold_df.empty:
    con.register('stage2_fold_df', stage2_fold_df)
    con.execute(f"COPY stage2_fold_df TO '{stage2_fold_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
    con.unregister('stage2_fold_df')

if RUN_REGRESSOR_TUNING or not stage3_tuning_df.empty:
    con.register('stage3_tuning_df', stage3_tuning_df)
    con.execute(f"COPY stage3_tuning_df TO '{stage3_tuning_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
    con.unregister('stage3_tuning_df')

if shap_summary_df is not None:
    con.register('shap_summary_df', shap_summary_df)
    con.execute(f"COPY shap_summary_df TO '{shap_summary_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
    con.unregister('shap_summary_df')

for name in [
    'stage1_threshold_df',
    'stage1_handoff_df',
    'stage2_comparison_df',
    'stage3_summary_df',
    'val_pipeline_predictions_df',
    'test_pipeline_predictions_df',
    'val_top5_recommendations_df',
    'test_top5_recommendations_df',
    'stage1_threshold_sweep_df',
    'stage2_tuning_df',
    'pipeline_summary_df',
]:
    con.unregister(name)

print(f'Saved stage 1 model to: {stage1_model_path}')
print(f'Saved stage 2 model to: {stage2_model_path}')
print(f'Saved stage 3 model to: {stage3_model_path}')
print(f'Saved pipeline summary to: {pipeline_summary_path}')
print(f'Saved stage 1 threshold summary to: {stage1_threshold_summary_path}')
print(f'Saved stage 1 handoff summary to: {stage1_handoff_summary_path}')
print(f'Saved ranking comparison to: {ranking_comparison_path}')
print(f'Saved regression summary to: {regression_summary_path}')
print(f'Saved validation pipeline predictions to: {val_pipeline_predictions_path}')
print(f'Saved test pipeline predictions to: {test_pipeline_predictions_path}')
print(f'Saved test top-5 recommendations to: {test_top5_recommendations_path}')
